# Runner (PLAN.md section 9)

One notebook for every logged run: only the `COMMAND` cell changes. Settings: GPU T4 x2, Internet on, input: your private `repo.zip` dataset.
Pin `COMMIT` to a hash for any number that goes in the report. Outputs land in `/kaggle/working/runs/<run_id>/`;
download them or save them as a **private** Kaggle Dataset after each run.

In [ ]:
# Code comes from a PRIVATE Kaggle dataset holding repo.zip, made on your laptop with:
#   git archive --format=zip --add-virtual-file=COMMIT:$(git rev-parse HEAD) -o repo.zip HEAD
# Add it with "Add Input". Upload a new version of the dataset whenever the code changes.
import shutil
from pathlib import Path
src = next(Path("/kaggle/input").rglob("src/common/schemas.py")).parents[2]
shutil.rmtree("/tmp/repo", ignore_errors=True)
shutil.copytree(src, "/tmp/repo")
%cd /tmp/repo
!cat COMMIT

In [ ]:
!pip install -q -U bitsandbytes peft accelerate
!pip install -q -r requirements.txt

In [ ]:
# Pick one. Each command logs config.json (commit, seed, platform, GPU, GPU hours), preds.jsonl and metrics.json.
MODEL = "Qwen/Qwen2.5-7B-Instruct"

# B1 (few-shot, no retrieval) on dev. Needs data/dev/items.jsonl and data/leases/*.json on main.
COMMAND = f"python -m src.eval.run --items data/dev/items.jsonl --leases data/leases --model {MODEL} --backend hf --mode none --out /kaggle/working/runs"

# Ablation arm 2: flat retrieval (needs src/retrieval from Saksham).
# COMMAND = f"python -m src.eval.run --items data/dev/items.jsonl --leases data/leases --model {MODEL} --backend hf --mode flat --out /kaggle/working/runs"

# 100-pair generation trial (TA point 2). Model and settings come from src/train/gen_config.json.
# COMMAND = "python -m src.train.gen_pairs --n 100 --out /kaggle/working/runs/gen-trial"

# QLoRA on kept pairs (learning curve: --limit 250 / 500 / 1000 / 2000).
# COMMAND = f"python -m src.train.qlora --model {MODEL} --data /kaggle/working/runs/gen-full/kept.jsonl --out /kaggle/working/runs/qlora-250 --limit 250"

In [ ]:
!{COMMAND}

In [ ]:
!cd /kaggle/working && zip -qr runs.zip runs && ls -lh runs.zip